In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, roc_auc_score
import os

path = os.getenv("DATA_PATH")
df = pd.read_csv(path)
target = "Problem_SKU"
seed = 1337 
# One-hot encode Storage_Size (drop size_4 as baseline)
size_dummies = pd.get_dummies(df['Storage_Size'], prefix='Size', drop_first=True).astype(int)

# Encode Defect_In_Linked_Receive as 0/1
defect_linked_num = df['Defect_In_Linked_Receive'].astype(int)

# Numeric features (keep standardized)
numeric_features = [
    "Global_SKU_Defect_Rate_%_std",
    "ABS_Volume_Difference_std",
    "Aisle_Hold_%_std",
    "#_Pick_Events_std",
    "#_Pick_Events_In_Clique_std",
    "#_Picks_std",
    "#_Picks_In_Clique_std",
    "Time_In_Loc_std",
    "Current_Max_Volume_std",
]

feature_cols = numeric_features + list(size_dummies.columns) + ['Defect_In_Linked_Receive']

# Combine all properly encoded features
X = df[numeric_features].copy()
X = pd.concat([X, size_dummies, defect_linked_num], axis=1)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=seed, stratify=y
)

dt = DecisionTreeClassifier(
    max_depth=None,        # let it grow; you can cap later
    min_samples_leaf=50,   # mild regularization
    random_state=seed,
)

dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_proba_dt = dt.predict_proba(X_test)[:, 1]

print("Decision Tree")
print(classification_report(y_test, y_pred_dt, digits=3))
print("ROC AUC:", roc_auc_score(y_test, y_proba_dt))

import numpy as np

fi_dt = pd.Series(dt.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(fi_dt)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_recall = DecisionTreeClassifier(
    class_weight='balanced',  # ← weights minority class higher
    min_samples_leaf=20,
    random_state=seed
)

dt_recall.fit(X_train, y_train)

y_pred_dt = dt_recall.predict(X_test)
y_proba_dt = dt_recall.predict_proba(X_test)[:, 1]

print("Decision Tree")
print(classification_report(y_test, y_pred_dt, digits=3))
print("ROC AUC:", roc_auc_score(y_test, y_proba_dt))

import numpy as np

fi_dt_recall = pd.Series(dt_recall.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(fi_dt_recall)